# 影视作品网络

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import scipy
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Heiti TC']
plt.rcParams['axes.unicode_minus'] = False
from networkx.algorithms import bipartite
import json

In [2]:
import sys
sys.path.append("..")

# 数据处理

In [3]:
df_raw = pd.read_csv("../data/movie_data_rated.csv")
df_raw['k_cast_id'] = df_raw['k_cast_id'].apply(lambda x: str(x))
df_raw

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,1017182,陈燕燕,女,10771202,演员,是,1,10771202,深闺疑云,电影,...,7.8,92,92.0,27.0,21542453,2034413,54,赵兰,m21542453,演员
1,1043845,刘志荣,男,10581318,演员,是,11,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2087739,75,NaN,m21162685,导演/演员
2,1050373,茅瑛,女,10573512,演员,是,1,10573512,鬼娘子,电影,...,5.8,235,235.0,37.0,21147073,2100795,79,NaN,m21147073,演员
3,1124360,利智,女,10581318,演员,是,3,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2248769,94,NaN,m21162685,演员
4,1314471,王挺,男,10531190,演员,是,1,10531190,惊天动地,电视剧,...,8.5,326,326.0,53.0,21062429,2628991,147,NaN,m21062429,导演/演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258252,1354509,林泳淘,女,26984991,演员,否,999,26984991,婚姻合伙人,电视剧,...,5.8,1259,1259.0,85.0,53970031,2709067,608916,邓秀玉,m53970031,演员
258253,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,7.0,211,211.0,38.0,4675243,54993117,608920,NaN,m4675243,演员
258254,27565771,董亚春,女,2270516,导演,是,2,2270516,八月一日,电影,...,6.4,1123,1123.0,85.0,4541081,55131591,608921,NaN,m4541081,导演
258255,35415094,张可,女,7053738,演员,否,999,7053738,樱桃,电视剧,...,5.7,2003,2003.0,107.0,14107525,70830237,608926,NaN,m14107525,演员


# 方法

In [4]:
# 生成图
def get_G(df):
    G = nx.from_pandas_edgelist(
        df,
        source='k_cast_id',
        target='movie_id_m',
        edge_attr=True,
        create_using=nx.Graph()
    )
    return G

In [5]:
# 最大连通子图
def get_largest_connected_component(G):
    if nx.is_connected(G):
        return G
    else:
        largest_cc = max(nx.connected_components(G), key=len)
        return G.subgraph(largest_cc).copy()

In [6]:
# 二分图，投影到电影节点
def get_movie_bip(G, cast_ids):
    return bipartite.projected_graph(G, G.nodes - cast_ids)

In [7]:
# 绘图
def draw_movie_net(G, k=None):
    pos = nx.spring_layout(G, k=k, seed=42)

    plt.figure(figsize=(10, 10))
    nx.draw_networkx(
        G,
        pos=pos,
        with_labels=False,
        node_size=1,
        edge_color="gainsboro",
        alpha=0.4,
    )
    plt.title("Movie Network")
    plt.axis('off')
    plt.show()
    print(G.number_of_nodes(), G.number_of_edges())

In [8]:
# 按评分提取数据
def get_lowest_rated_movie_casts(df, work_type="电影", ascending=True, top_n=100):
    df_type = df.loc[(df['k_type'] == work_type).copy()]
    df_type_simple = df_type[['movie_id_m', 'k_title',
                   'rating_num', 'k_rating_people']].drop_duplicates().reset_index(drop=True)
    df_type_n = df_type_simple.sort_values(by='rating_num', ascending=ascending).head(top_n)
    df_n = df_type_n[['movie_id_m']].merge(df,
                                            left_on='movie_id_m',
                                            right_on='movie_id_m',
                                            how='left')
    return df_n

In [9]:
# 作品筛选
def works_filter(df, work_type="电影", sort_by='rating_num', ascending=True, top_n=100, rating_people_min=0):
    df_type = df.loc[(df['k_type'] == work_type).copy()] if work_type else df
    df_type_simple = df_type[['movie_id_m', 'k_title',
                   'rating_num', 'k_rating_people']].drop_duplicates().reset_index(drop=True)
    df_type_n = df_type_simple[df_type_simple['k_rating_people'] >= rating_people_min].sort_values(by=sort_by, ascending=ascending).head(top_n)
    df_n = df_type_n[['movie_id_m']].merge(df_type,
                                            left_on='movie_id_m',
                                            right_on='movie_id_m',
                                            how='left')
    return df_n

In [10]:
# 度数为1的影人节点
def degree_1_cast_ids(G):
    degree_1_nodes = [node for node, degree in G.degree() if degree == 1]
    nodes_to_remove = [i for i in degree_1_nodes if 'm' not in str(i)]
    return nodes_to_remove

In [11]:
# 生成图表数据
def get_chart_data(G, df, sub_pos=False, work_type="电影"):
    nodes = G.nodes()
    n = len(nodes)
    k = None if n < 500 else 1.5
    pos = nx.kamada_kawai_layout(G) if sub_pos else nx.spring_layout(G, k=k, seed=42)
    # pos= nx.kamada_kawai_layout(G)
    # pos = nx.forceatlas2_layout(G)
    degrees = dict(G.degree())
    cast_degrees = {k: v for k, v in degrees.items() if 'm' not in str(k)}
    # 将cast_degrees归一到2-7的范围
    cast_degree_values = np.array(list(cast_degrees.values()))
    min_degree = cast_degree_values.min()
    max_degree = cast_degree_values.max()
    for key in cast_degrees:
        cast_degrees[key] = 2 + 5 * (cast_degrees[key] -
                                     min_degree) / (max_degree - min_degree) 
    # 二分图-电影
    movie_nodes = [n for n in nodes if 'm' in str(n)]
    G_movie = bipartite.projected_graph(G, movie_nodes)
    movie_degrees = dict(G_movie.degree())
    # 将movie_degrees归一到2-9的范围
    movie_degree_values = np.array(list(movie_degrees.values()))
    min_degree = movie_degree_values.min()
    max_degree = movie_degree_values.max()
    for key in movie_degrees:
        movie_degrees[key] = 2 + 7 * (movie_degrees[key] -
                                      min_degree) / (max_degree - min_degree)
    # 节点属性
    nodes_data = []
    movie_num = 0
    for node in nodes:
        node_pos = pos[node]
        node_data = {}
        if 'm' in str(node):
            node_data['category'] = work_type if work_type else df[df['movie_id_m'] ==
                                    node]['k_type'].values[0]
            node_data['title'] = df[df['movie_id_m'] ==
                                    node]['k_title'].values[0]
            node_data['degree'] = movie_degrees.get(node, 0)
            node_data['order'] = 0 if work_type else (0 if node_data['category'] == '电影' else 1)
            movie_num += 1
        else:
            node_data['category'] = '导演/演员'
            node_data['title'] = df[df['k_cast_id'] ==
                                    node]['cast_name'].values[0]
            node_data['degree'] = cast_degrees.get(node, 0)
            node_data['order'] = 1 if work_type else 2
        node_data['id'] = str(node)
        node_data['x'] = round(float(node_pos[0]), 6)
        node_data['y'] = round(float(node_pos[1]), 6)
        nodes_data.append(node_data)
    # 边属性
    edges = G.edges()
    edges_data = [{
        'source': str(s),
        'target': str(t),
        'weight': 1
    } for s, t in edges]

    cast_num = len(nodes) - movie_num
    if work_type is None:
        subtitle = f"{movie_num}部影视作品, {cast_num}位导演与主要演员"
        categories = [{
            'name': '电影',
            'order': 0
        }, {
            'name': '电视剧',
            'order': 1
        }, {
            'name': '导演/演员',
            'order': 2
        }]
    else:
        subtitle = f"{movie_num}部{work_type}, {cast_num}位导演与主要演员"
        categories = [{
            'name': work_type,
            'order': 0
        }, {
            'name': '导演/演员',
            'order': 1
        }]
    net_data = {    
        'nodes': nodes_data,
        'edges': edges_data,
        'categories': categories,
        'subtitle': subtitle,
    }
    return net_data

In [12]:
# 生成电影二分图图表数据
def get_movie_bip_chart_data(G, df):
    nodes = G.nodes()
    # 二分图-电影
    movie_nodes = [n for n in nodes if 'm' in str(n)]
    G_movie = bipartite.weighted_projected_graph(G, movie_nodes)
    movie_degrees = dict(G_movie.degree())
    # 将movie_degrees归一到2-9的范围
    movie_degree_values = np.array(list(movie_degrees.values()))
    min_degree = movie_degree_values.min()
    max_degree = movie_degree_values.max()
    for key in movie_degrees:
        movie_degrees[key] = 2 + 7 * (movie_degrees[key] -
                                      min_degree) / (max_degree - min_degree)
    pos = nx.spring_layout(G_movie, k=3, scale=1, seed=412)
    nodes_data = []
    for node in movie_nodes:
        node_data = {}
        node_pos = pos[node]
        node_data['category'] = '电影'
        node_data['title'] = df[df['movie_id_m'] == node]['k_title'].values[0]
        node_data['degree'] = round(float(movie_degrees.get(node, 0)), 2)
        node_data['order'] = 0
        node_data['id'] = str(node)
        node_data['x'] = round(float(node_pos[0]), 6)
        node_data['y'] = round(float(node_pos[1]), 6)
        nodes_data.append(node_data)
    edges = G_movie.edges(data=True)
    edges_data = [{
        'source': str(s),
        'target': str(t),
        'weight': d['weight']
    } for s, t, d in edges]
    movie_num = len(movie_nodes)
    subtitle = f"{movie_num}部电影的关联关系"
    net_data = {
        'nodes': nodes_data,
        'edges': edges_data,
        'categories': [{
            'name': '电影',
            'order': 0
        }],
        'subtitle': subtitle,
    }
    return net_data

In [13]:
def get_chart_data_all(df, title_prefix, work_type="电影"):
    # 整图
    G_n = get_G(df)
    # 最大子图
    G_n_largest = get_largest_connected_component(G_n)
    movie_bip_data = get_movie_bip_chart_data(G_n_largest, df)
    # 去除度为1影人节点
    G_n_node_to_remove = degree_1_cast_ids(G_n)
    G_n_d2 = G_n.copy()
    G_n_d2.remove_nodes_from(G_n_node_to_remove)
    # 最大子图去除度为1影人节点
    G_n_largest_node_to_remove = degree_1_cast_ids(G_n_largest)
    G_n_d2_largest = G_n_largest.copy()
    G_n_d2_largest.remove_nodes_from(G_n_largest_node_to_remove)

    # 生成图表数据
    net_data = get_chart_data(G_n, df, work_type=work_type)
    net_data['title'] = title_prefix + "-全量数据"
    net_data_largest = get_chart_data(G_n_largest, df, work_type=work_type)
    net_data_largest['title'] = title_prefix + "-最大连通子图"
    net_data_d2 = get_chart_data(G_n_d2, df, sub_pos=True, work_type=work_type)
    net_data_d2['title'] = title_prefix + "-去除度为1的影人节点"
    net_data_d2_largest = get_chart_data(G_n_d2_largest, df, sub_pos=True, work_type=work_type)
    net_data_d2_largest['title'] = title_prefix + "-最大连通子图去除度为1的影人节点"
    net_data = {
        "net_data": net_data,
        "net_data_largest": net_data_largest,
        "net_data_d2": net_data_d2,
        "net_data_d2_largest": net_data_d2_largest,
        "movie_bip_data": movie_bip_data
    }
    return net_data

In [14]:
top_desc = """
主流的电影研究往往集中在高分经典上——演员的奖项、导演的稳定创作力、以及票房表现等传统指标。
但我在想，既然好片能养出“黄金合作班底”，那这些经典佳作之间，是否也隐藏着某种合作网络？
于是，我从豆瓣中选取了
[向右R][向右R]评分人数超过10万人且评分最高的200部电影，
这批作品的分数高得离谱：最低都在 7.6分以上，最顶尖的甚至稳居影史榜单多年不动。
围绕这 200 部影片的 2700 位导演与主要演员（影人），我以“影人—电影”为连边关系，构建了一个无向图模型，进而得到了这批高分电影的合作关系网络，也就是下面的四张图。
[一R]图 1：
高分电影关系网（完整网络）
包含全部 200 部高分电影及其导演与主演。
可以看到整体结构相对庞大，包含多个连通子图。
有些子图像散落在影史角落的“小宇宙”，彼此之间并不直接关联。
如果你眼尖，说不定还能在图里找出某些你熟悉的组合。
[二R]图 2：
高分电影关系网（最大连通子图）
当我们剔除那些完全孤立的子图后，剩下的最大连通子图变得更加紧密，该子图包含175部电影及2574位影人。
不过仍然可以看到不少 “度为 1” 的影人节点——这意味着他们只在这 200 部电影中参与了一部作品。
为了让合作模式更清晰，我进一步剔除了这些度为 1 的节点，得到下方更集中的网络结构。
[三R]图 3：
高分电影关系网（去除度为1影人），此时剩余545位影人，而电影仍为200部。
可以看到部分电影节点变成孤立点，说明它们的主创人员并未与其他高分电影形成交叉合作。
但整体结构的形态开始显现，影史中那些频繁合作的黄金阵容也开始浮出水面。
[四R]图 4：
高分电影关系网（最大连通子图 + 去除度为1影人）
这是整个结构最紧凑、也最具“合作意义”的部分，此时剩余175部电影及539位影人。
影人之间的联系在这里最为密切，能看出哪些导演常与固定班底合作，以及哪些演员频繁在高质量项目中相遇。
研究这些高分电影的合作网络，能从另一种角度理解“经典是如何诞生的”：
不仅是导演与演员的个人实力，更是长期合作、默契搭档，以及稳定可靠的创作关系网在背后支撑。
当然，他们并不一定只拍佳作，高分电影只是他们职业轨迹上的亮眼部分。但正是这些高分作品，构成了他们在电影史中的共同坐标。
[向右R][向右R]特别声明：
数据来源于公开的网络数据，旨在探索电影合作网络的结构特征，难免存在遗漏或错误，内容仅限学习与研究使用。
#数据可视化 #经典电影推荐 #与电影对视120次 
"""

# main

In [15]:
df_raw 

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,1017182,陈燕燕,女,10771202,演员,是,1,10771202,深闺疑云,电影,...,7.8,92,92.0,27.0,21542453,2034413,54,赵兰,m21542453,演员
1,1043845,刘志荣,男,10581318,演员,是,11,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2087739,75,NaN,m21162685,导演/演员
2,1050373,茅瑛,女,10573512,演员,是,1,10573512,鬼娘子,电影,...,5.8,235,235.0,37.0,21147073,2100795,79,NaN,m21147073,演员
3,1124360,利智,女,10581318,演员,是,3,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2248769,94,NaN,m21162685,演员
4,1314471,王挺,男,10531190,演员,是,1,10531190,惊天动地,电视剧,...,8.5,326,326.0,53.0,21062429,2628991,147,NaN,m21062429,导演/演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258252,1354509,林泳淘,女,26984991,演员,否,999,26984991,婚姻合伙人,电视剧,...,5.8,1259,1259.0,85.0,53970031,2709067,608916,邓秀玉,m53970031,演员
258253,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,7.0,211,211.0,38.0,4675243,54993117,608920,NaN,m4675243,演员
258254,27565771,董亚春,女,2270516,导演,是,2,2270516,八月一日,电影,...,6.4,1123,1123.0,85.0,4541081,55131591,608921,NaN,m4541081,导演
258255,35415094,张可,女,7053738,演员,否,999,7053738,樱桃,电视剧,...,5.7,2003,2003.0,107.0,14107525,70830237,608926,NaN,m14107525,演员


In [16]:
work_type = '电视剧'
work_type_value = 'movie' if work_type == '电影' else 'tv'
sort_by = 'rating_num'
ascending = False
asc_value = 'min' if ascending else 'max'
title_value = '低分' if ascending else '高分'
top_n = 200
rating_people_min = 30000
df_filtered = works_filter(df_raw, work_type=work_type, sort_by=sort_by, ascending=ascending, top_n=top_n, rating_people_min=rating_people_min)
df_filtered

,movie_id_m,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,cast_role_agg
0,m2883637,27483006,韩再芬,女,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,54966061,27706,沈玉英,导演/演员
1,m2883637,30414323,洪宗义,男,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,60828695,35489,方伯谦,演员
2,m2883637,27483118,马仑,男,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,54966285,58588,段祺瑞,演员
3,m2883637,27562254,刘伟明,男,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,55124557,65556,张謇,导演/演员
4,m2883637,27482432,赵立新,男,1441794,演员,否,999,1441794,走向共和,...,有评分,9.7,73275,73275.0,843.0,2883637,54964913,71786,罗文,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8446,m69559601,1381885,黄馨瑶,女,34779776,演员,否,999,34779776,苍兰诀,...,有评分,8.1,650830,650830.0,2296.0,69559601,2763819,456219,三生,演员
8447,m69559601,27575553,李一桐,女,34779776,演员,是,6,34779776,苍兰诀,...,有评分,8.1,650830,650830.0,2296.0,69559601,55151155,488968,司命,演员
8448,m69559601,35065662,程梓,女,34779776,演员,否,999,34779776,苍兰诀,...,有评分,8.1,650830,650830.0,2296.0,69559601,70131373,508900,蝶衣,演员
8449,m69559601,27573708,王鹤棣,男,34779776,演员,是,2,34779776,苍兰诀,...,有评分,8.1,650830,650830.0,2296.0,69559601,55147465,516732,东方青苍,演员


In [17]:
df_info = df_filtered[['k_movie_id', 'k_title', 'rating_num', 'k_movie_year', 'k_rating_people', 'k_type']].drop_duplicates().sort_values(by='rating_num', ascending=ascending).reset_index(drop=True)
df_info['k_movie_id'] = df_info['k_movie_id'].apply(lambda x: str(x))
df_info.to_json(f'json_data/works_info_{work_type_value}_{sort_by}_{asc_value}_{top_n}_{rating_people_min}.json', orient='records', force_ascii=False, indent=4)
df_info

,k_movie_id,k_title,rating_num,k_movie_year,k_rating_people,k_type
0,2883637,走向共和,9.7,2003,73275.0,电视剧
1,4420051,大明王朝1566,9.7,2007,145676.0,电视剧
2,3729669,红楼梦,9.7,1987,168190.0,电视剧
3,53207743,毛骗 终结篇,9.7,2015,68590.0,电视剧
4,7765479,武林外传,9.6,2006,400757.0,电视剧
...,...,...,...,...,...,...
195,70406757,爱情而已,8.2,2023,156889.0,电视剧
196,53861129,无证之罪,8.1,2017,228734.0,电视剧
197,71451761,尘封十三载,8.1,2023,142058.0,电视剧
198,53532477,风起陇西,8.1,2022,101385.0,电视剧


In [18]:
net_data = get_chart_data_all(df_filtered, title_prefix=f"{title_value}{work_type}关系网络", work_type=work_type)
# 保存处理后的数据
with open(f'json_data/net_data_{work_type_value}_{sort_by}_{asc_value}_{top_n}_{rating_people_min}.json', 'w', encoding='utf-8') as f:
    json.dump(net_data, f, ensure_ascii=False, indent=4)

In [19]:
df_raw

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,1017182,陈燕燕,女,10771202,演员,是,1,10771202,深闺疑云,电影,...,7.8,92,92.0,27.0,21542453,2034413,54,赵兰,m21542453,演员
1,1043845,刘志荣,男,10581318,演员,是,11,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2087739,75,NaN,m21162685,导演/演员
2,1050373,茅瑛,女,10573512,演员,是,1,10573512,鬼娘子,电影,...,5.8,235,235.0,37.0,21147073,2100795,79,NaN,m21147073,演员
3,1124360,利智,女,10581318,演员,是,3,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2248769,94,NaN,m21162685,演员
4,1314471,王挺,男,10531190,演员,是,1,10531190,惊天动地,电视剧,...,8.5,326,326.0,53.0,21062429,2628991,147,NaN,m21062429,导演/演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258252,1354509,林泳淘,女,26984991,演员,否,999,26984991,婚姻合伙人,电视剧,...,5.8,1259,1259.0,85.0,53970031,2709067,608916,邓秀玉,m53970031,演员
258253,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,7.0,211,211.0,38.0,4675243,54993117,608920,NaN,m4675243,演员
258254,27565771,董亚春,女,2270516,导演,是,2,2270516,八月一日,电影,...,6.4,1123,1123.0,85.0,4541081,55131591,608921,NaN,m4541081,导演
258255,35415094,张可,女,7053738,演员,否,999,7053738,樱桃,电视剧,...,5.7,2003,2003.0,107.0,14107525,70830237,608926,NaN,m14107525,演员


# 专题分析

## 2025年

In [20]:
movie_year = 2025
df_year = df_raw[df_raw['k_movie_year'] == movie_year]
df_year

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
1754,27219409,姜大卫,男,30174089,演员,是,9,30174089,内幕,电影,...,6.3,11475,11475.0,269.0,60348227,54438867,3536,NaN,m60348227,导演/演员
1795,33411105,牛超,男,35873709,演员,是,10,35873709,刑警的日子,电视剧,...,6.9,8483,8483.0,242.0,71747467,66822259,3625,夏宇,m71747467,演员
1821,34901029,沐桐,男,37054291,演员,否,999,37054291,锦绣芳华,电视剧,...,7.0,229121,229121.0,1266.0,74108631,69802107,3682,射雁,m74108631,演员
1843,27558459,白润音,男,33437319,演员,是,15,33437319,我们与恶的距离2,电视剧,...,8.4,27399,27399.0,480.0,66874687,55116967,3731,罗誉,m66874687,演员
1874,27480863,钱泳辰,男,36455616,演员,是,22,36455616,赴山海,电视剧,...,5.9,433940,433940.0,1600.0,72911281,54961775,3810,孟相逢,m72911281,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258082,27566467,孔二狗,男,36638134,导演,是,1,36638134,老舅,电视剧,...,7.1,20550,20550.0,382.0,73276317,55132983,608532,NaN,m73276317,导演/演员
258085,27567348,杜江,男,36296619,演员,是,16,36296619,志愿军：浴血和平,电影,...,7.1,55796,55796.0,629.0,72593287,55134745,608540,崔建功,m72593287,演员
258124,30390143,白宇帆,男,26749938,演员,否,999,26749938,酱园弄·悬案,电影,...,5.7,171169,171169.0,988.0,53499925,60780335,608647,NaN,m53499925,演员
258135,34572275,韩秀一,男,35693315,演员,否,999,35693315,北上,电视剧,...,7.0,131970,131970.0,961.0,71386679,69144599,608663,强哥,m71386679,演员


In [21]:
# 作品数量
works_num = df_year['movie_id_m'].nunique()
movie_num = df_year[df_year['k_type'] == '电影']['movie_id_m'].nunique()
tv_num = df_year[df_year['k_type'] == '电视剧']['movie_id_m'].nunique()
works_num, movie_num, tv_num

(239, 114, 125)

In [22]:
net_data_year = get_chart_data_all(df_year, title_prefix=f"{movie_year}年华语影视关系网络", work_type=None)
# 保存处理后的数据
with open(f'json_data/net_data_{movie_year}.json', 'w', encoding='utf-8') as f:
    json.dump(net_data_year, f, ensure_ascii=False, indent=4)

In [23]:
# 生成一段 df_year 数据描述
description_year = f"""2025年华语影视作品关系网络包含 {works_num} 部影视作品，其中电影 {movie_num} 部，电视剧 {tv_num} 部。围绕这些作品的导演与主要演员，共有 {len(df_year['k_cast_id'].unique())} 位，他们共同构成了这个复杂而丰富的影视关系网络。"""

description_year

'2025年华语影视作品关系网络包含 239 部影视作品，其中电影 114 部，电视剧 125 部。围绕这些作品的导演与主要演员，共有 4008 位，他们共同构成了这个复杂而丰富的影视关系网络。'

# 特定影人

In [34]:
k_cast_id = "55130387"

In [35]:
# 包含k_cast_id的所有movie_id_m，含这些movie_id_m的所有记录
df_cast_movies = df_raw[df_raw['k_cast_id'] == k_cast_id][['movie_id_m']].drop_duplicates()
df_cast_related = df_raw.merge(df_cast_movies,
                              left_on='movie_id_m',
                              right_on='movie_id_m',
                              how='inner')
df_cast_related

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,30301886,王冬,男,20431716,演员,是,4,20431716,男人的刀,电视剧,...,7.6,147,147.0,33.0,40863481,60603821,586,NaN,m40863481,演员
1,27550212,汪小壹,女,26889177,演员,是,10,26889177,异物志,电视剧,...,8.4,60664,60664.0,714.0,53778403,55100473,3914,NaN,m53778403,导演/演员
2,27488967,王秀月,女,5250330,演员,否,999,5250330,快乐的小2B,电视剧,...,7.6,707,707.0,73.0,10500709,54977983,6322,NaN,m10500709,演员
3,27550913,包志强,男,5250330,演员,否,999,5250330,快乐的小2B,电视剧,...,7.6,707,707.0,73.0,10500709,55101875,6721,NaN,m10500709,演员
4,30161546,车志刚,男,5250330,演员,否,999,5250330,快乐的小2B,电视剧,...,7.6,707,707.0,73.0,10500709,60323141,8447,NaN,m10500709,导演/演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253,27567442,赵宁,女,26603847,演员,否,999,26603847,毛骗 终结篇,电视剧,...,9.7,68590,68590.0,816.0,53207743,55134933,600837,NaN,m53207743,演员
254,34930412,李文帅,男,36382628,演员,否,999,36382628,海市蜃楼,电视剧,...,8.1,9664,9664.0,280.0,72765305,69860873,601181,耿师爷,m72765305,演员
255,27503136,韩栋,男,36424178,演员,是,1,36424178,真相背后,电视剧,...,6.1,3080,3080.0,137.0,72848405,55006321,602260,NaN,m72848405,演员
256,35201466,付余,女,33438451,演员,否,999,33438451,城市的边缘,电视剧,...,8.4,40382,40382.0,582.0,66876951,70402981,602611,NaN,m66876951,演员


In [36]:
df_cast_related['movie_id_m'].nunique()

18

In [37]:
df_cast_related['k_title'].unique()

array(['男人的刀', '异物志', '快乐的小2B', '冤家宜解不宜结', '金牌保镖', '毛骗 终结篇', '城市的边缘',
       '输不起', '毛骗 第二季', '真相背后', '麻辣宿舍', '毛骗 第一季', '海市蜃楼', '杀不死', '一屋高才生',
       '麻辣兄弟之疯狂一夜', '非常保镖', '第三个人'], dtype=object)

# 数据统计

In [28]:
top_n = 200
df_movie_top_n = get_lowest_rated_movie_casts(df_raw,
                                              ascending=False,
                                              top_n=top_n)
df_movie_top_n

,movie_id_m,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,cast_role_agg
0,m2583141,1003494,张国荣,男,1291546,演员,是,1,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,2007037,388,程蝶衣(小豆子,导演/演员
1,m2583141,27226213,蒋雯丽,女,1291546,演员,是,6,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,54452475,1781,小豆子生母,导演/演员
2,m2583141,27256028,张丰毅,男,1291546,演员,是,2,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,54512105,3236,段小楼(小石头,演员
3,m2583141,27206595,葛优,男,1291546,演员,是,4,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,54413239,3384,袁世卿(袁四爷,演员
4,m2583141,1275231,李春,男,1291546,演员,是,14,1291546,霸王别姬,...,有评分,9.6,1783862,1783862.0,4138.0,2583141,2550511,8930,少年小四,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2073,m3924193,1015616,张石川,男,1962072,导演,是,1,1962072,歌女红牡丹,...,有评分,8.6,88,88.0,28.0,3924193,2031281,44223,NaN,导演/演员
2074,m3924193,1317484,王献斋,男,1962072,演员,是,3,1962072,歌女红牡丹,...,有评分,8.6,88,88.0,28.0,3924193,2635017,84909,NaN,导演/演员
2075,m3924193,27487368,胡蝶,女,1962072,演员,是,2,1962072,歌女红牡丹,...,有评分,8.6,88,88.0,28.0,3924193,54974785,156518,NaN,导演/演员
2076,m3924193,1036006,龚稼农,男,1962072,演员,是,1,1962072,歌女红牡丹,...,有评分,8.6,88,88.0,28.0,3924193,2072061,219685,NaN,导演/演员


In [29]:
df_movie_top_n_asc = get_lowest_rated_movie_casts(df_raw,
                                              ascending=True,
                                              top_n=top_n)
df_movie_top_n_asc

,movie_id_m,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,cast_role_agg
0,m51454521,27494382,刘萌萌,女,25727236,演员,是,5,25727236,201413,...,有评分,2.1,1413,1413.0,54.0,51454521,54988813,32081,NaN,演员
1,m51454521,27483544,夏一瑶,女,25727236,演员,是,6,25727236,201413,...,有评分,2.1,1413,1413.0,54.0,51454521,54967137,42422,NaN,演员
2,m51454521,27573296,方品乔,女,25727236,演员,否,999,25727236,201413,...,有评分,2.1,1413,1413.0,54.0,51454521,55146641,73730,NaN,演员
3,m51454521,27551274,彭波,男,25727236,演员,是,8,25727236,201413,...,有评分,2.1,1413,1413.0,54.0,51454521,55102597,184208,NaN,演员
4,m51454521,27552814,沈婷婷,女,25727236,演员,是,7,25727236,201413,...,有评分,2.1,1413,1413.0,54.0,51454521,55105677,204070,NaN,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1208,m71326273,1316213,何佩瑜,女,35663112,演员,是,5,35663112,封神：妲己,...,有评分,2.7,1522,1522.0,64.0,71326273,2632475,125760,NaN,演员
1209,m71326273,30382866,刘迪洋,男,35663112,导演,是,1,35663112,封神：妲己,...,有评分,2.7,1522,1522.0,64.0,71326273,60765781,142680,NaN,导演
1210,m71326273,27549407,沈震轩,男,35663112,演员,是,3,35663112,封神：妲己,...,有评分,2.7,1522,1522.0,64.0,71326273,55098863,326254,NaN,演员
1211,m71326273,27210829,钟欣潼,女,35663112,演员,是,1,35663112,封神：妲己,...,有评分,2.7,1522,1522.0,64.0,71326273,54421707,369921,NaN,演员


In [30]:
# 计数
def get_df_stats(df):
    # 作品数量
    works_count = df['movie_id_m'].nunique()
    # 导演数量
    director_count = df[df['k_role'] == '导演']['k_cast_id'].nunique()
    # 演员数量
    actor_count = df[df['k_role'] == '演员']['k_cast_id'].nunique()
    # 影人数量
    cast_count = df['k_cast_id'].nunique()
    # 最高分
    rating_max = df['rating_num'].max()
    # 最低分
    rating_min = df['rating_num'].min()
    # 平均分
    rating_mean = df['rating_num'].mean()
    return {
        'works_count': works_count,
        'director_count': director_count,
        'actor_count': actor_count,
        'cast_count': cast_count,
        'rating_max': rating_max,
        'rating_min': rating_min,
        'rating_mean': round(rating_mean, 2),
        'rating_people_num': '10W+'
    }

In [31]:
# 图指标
def get_graph_metrics(G, df, is_bipartite=False):
    metrics = {}
    # 度
    degrees = dict(G.degree())
    # 节点数
    metrics['num_nodes'] = G.number_of_nodes()
    # 边数
    metrics['num_edges'] = G.number_of_edges()
    # 介数中心性最大值节点
    betweenness = nx.betweenness_centrality(G)

    if is_bipartite:
        # 最大度
        max_degree = max(degrees.values())
        metrics['max_degree'] = max_degree
        if max_degree > 1:
            # 最大度节点
            max_degree_node = [n for n in G.nodes() if degrees[n] == max_degree]
            # metrics['max_degree_node'] = max_degree_node
            # 最大度节点名称
            max_degree_node_names = []
            for n in max_degree_node:
                if 'm' in str(n):
                    name = df[df['movie_id_m'] == n]['k_title'].unique().tolist()
                else:
                    name = df[df['k_cast_id'] == n]['cast_name'].unique().tolist()
                max_degree_node_names.extend(name)
            metrics['max_degree_node_names'] = ",".join(max_degree_node_names)
        # 最大介数中心性
        max_betweenness = max(betweenness.values())
        metrics['max_betweenness'] = round(max_betweenness, 6)
        if max_betweenness > 0:
            # 最大介数中心性节点
            max_betweenness_node = [n for n in G.nodes() if betweenness[n] == max_betweenness]
            # metrics['max_betweenness_node'] = max_betweenness_node
            # 最大介数中心性节点名称
            max_betweenness_node_names = []
            for n in max_betweenness_node:
                if 'm' in str(n):
                    name = df[df['movie_id_m'] == n]['k_title'].unique().tolist()
                else:
                    name = df[df['k_cast_id'] == n]['cast_name'].unique().tolist()
                max_betweenness_node_names.extend(name)
            metrics['max_betweenness_node_names'] = ",".join(max_betweenness_node_names)
    else:
        # 影人节点
        cast_nodes = [n for n in G.nodes() if 'm' not in str(n)]
        # 影人节点数
        metrics['num_cast_nodes'] = len(cast_nodes)
        # 影人节点最大度
        max_cast_degree = max([degrees[n] for n in cast_nodes])
        metrics['max_cast_degree'] = max_cast_degree
        if max_cast_degree > 1:
            # 最大度影人节点
            max_cast_degree_node = [n for n in cast_nodes if degrees[n] == max_cast_degree]
            # 最大度影人节点名称
            max_cast_degree_node_names = df[df['k_cast_id'].isin(max_cast_degree_node)]['cast_name'].unique().tolist()
            # metrics['max_cast_degree_node'] = max_cast_degree_node
            metrics['max_cast_degree_node_names'] = ",".join(max_cast_degree_node_names)
        # 介数中心性最大值影人节点
        max_cast_betweenness = max([betweenness[n] for n in cast_nodes])
        metrics['max_cast_betweenness'] = round(max_cast_betweenness, 6)
        if max_cast_betweenness > 0:
            # 最大介数中心性影人节点
            max_cast_betweenness_node = [n for n in cast_nodes if betweenness[n] == max_cast_betweenness]
            # 最大介数中心性影人节点名称
            max_cast_betweenness_node_names = df[df['k_cast_id'].isin(max_cast_betweenness_node)]['cast_name'].unique().tolist()
            # metrics['max_cast_betweenness_node'] = max_cast_betweenness_node
            metrics['max_cast_betweenness_node_names'] = ",".join(max_cast_betweenness_node_names)

        # 电影节点
        movie_nodes = [n for n in G.nodes() if 'm' in str(n)]
        # 电影节点数
        metrics['num_movie_nodes'] = len(movie_nodes)
        # 电影节点最大度
        max_movie_degree = max([degrees[n] for n in movie_nodes])
        metrics['max_movie_degree'] = max_movie_degree
        if max_movie_degree > 1:
            # 最大度电影节点
            max_movie_degree_node = [n for n in movie_nodes if degrees[n] == max_movie_degree]
            # 最大度电影节点名称
            max_movie_degree_node_names = df[df['movie_id_m'].isin(max_movie_degree_node)]['k_title'].unique().tolist()
            # metrics['max_movie_degree_node'] = max_movie_degree_node
            metrics['max_movie_degree_node_names'] = ",".join(max_movie_degree_node_names)
        # 最大介数中心性电影节点
        max_movie_betweenness = max([betweenness[n] for n in movie_nodes])
        metrics['max_movie_betweenness'] = round(max_movie_betweenness, 6)
        if max_movie_betweenness > 0:
            # 最大介数中心性电影节点
            max_movie_betweenness_node = [n for n in movie_nodes if betweenness[n] == max_movie_betweenness]
            # 最大介数中心性电影节点名称
            max_movie_betweenness_node_names = df[df['movie_id_m'].isin(max_movie_betweenness_node)]['k_title'].unique().tolist()
            # metrics['max_movie_betweenness_node'] = max_movie_betweenness_node
            metrics['max_movie_betweenness_node_names'] = ",".join(max_movie_betweenness_node_names)

    # 密度
    metrics['density'] = round(nx.density(G), 6)
    # 平均聚类系数
    # metrics['average_clustering'] = nx.average_clustering(G)
    # 是否连通图
    metrics['is_connected'] = "是" if nx.is_connected(G) else "否"
    # 直径（仅适用于连通图）
    if nx.is_connected(G):
        metrics['diameter'] = nx.diameter(G)
    else:
        # metrics['diameter'] = None
        # 子图数量
        metrics['num_connected_components'] = nx.number_connected_components(G)
        # 最大子图
        largest_cc = max(nx.connected_components(G), key=len)
        G_largest = G.subgraph(largest_cc).copy()
        metrics['largest_cc_num_nodes'] = G_largest.number_of_nodes()
        metrics['largest_cc_num_nodes_ratio'] = f"{round(G_largest.number_of_nodes() / G.number_of_nodes() * 100, 2)}%"
        metrics['largest_cc_num_edges'] = G_largest.number_of_edges()
        metrics['largest_cc_num_edges_ratio'] = f"{round(G_largest.number_of_edges() / G.number_of_edges() * 100, 2)}%"

    return metrics

In [32]:
# main
def movie_net_stats(df):
    G = get_G(df)
    stats = {}
    stats['df_stats'] = get_df_stats(df)
    stats['graph_metrics'] = get_graph_metrics(G, df)
    if stats['graph_metrics']['is_connected'] == "否":
        largest_cc = max(nx.connected_components(G), key=len)
        stats['largest_connected_component'] = get_graph_metrics(G.subgraph(largest_cc).copy(), df)
    # 二分图-电影
    movie_nodes = [n for n in G.nodes() if 'm' in str(n)]
    G_movie = bipartite.projected_graph(G, movie_nodes)
    stats['movie_bipartite_graph_metrics'] = get_graph_metrics(G_movie, df, is_bipartite=True)
    # 最大连通子图二分图-电影
    if not nx.is_connected(G_movie):
        largest_cc = max(nx.connected_components(G_movie), key=len)
        stats['movie_bipartite_largest_connected_component'] = get_graph_metrics(G_movie.subgraph(largest_cc).copy(), df, is_bipartite=True)
    # 二分图-影人
    cast_nodes = [n for n in G.nodes() if 'm' not in str(n)]
    G_cast = bipartite.projected_graph(G, cast_nodes)
    stats['cast_bipartite_graph_metrics'] = get_graph_metrics(G_cast, df, is_bipartite=True)
    # 最大连通子图二分图-影人
    if not nx.is_connected(G_cast):
        largest_cc = max(nx.connected_components(G_cast), key=len)
        stats['cast_bipartite_largest_connected_component'] = get_graph_metrics(G_cast.subgraph(largest_cc).copy(), df, is_bipartite=True)
    return stats


In [33]:
year_stats = movie_net_stats(df_year)
year_stats

KeyboardInterrupt: 

In [ ]:
top_stats = movie_net_stats(df_movie_top_n)

In [ ]:
top_stats_asc = movie_net_stats(df_movie_top_n_asc)

In [ ]:
# 中英文对照字典
key_to_chinese = {
    'works_count': '作品数量',
    'director_count': '导演数量',
    'actor_count': '演员数量',
    'cast_count': '影人数量',
    'rating_max': '最高评分',
    'rating_min': '最低评分',
    'rating_mean': '平均评分',
    'rating_people_num': '评分人数标准',
    'num_nodes': '节点数',
    'num_edges': '边数',
    'max_degree': '最大度',
    'max_degree_node': '最大度节点',
    'max_degree_node_names': '最大度节点名称',
    'max_betweenness': '最大介数中心性',
    'max_betweenness_node': '最大介数中心性节点',
    'max_betweenness_node_names': '最大介数中心性节点名称',
    'num_cast_nodes': '影人节点数',
    'max_cast_degree': '影人最大度',
    'max_cast_degree_node': '影人最大度节点',
    'max_cast_degree_node_names': '影人最大度节点名称',
    'max_cast_betweenness': '影人最大介数中心性',
    'max_cast_betweenness_node': '影人最大介数中心性节点',
    'max_cast_betweenness_node_names': '影人最大介数中心性节点名称',
    'num_movie_nodes': '作品节点数',
    'max_movie_degree': '作品最大度',
    'max_movie_degree_node': '作品最大度节点',
    'max_movie_degree_node_names': '作品最大度节点名称',
    'max_movie_betweenness': '作品最大介数中心性',
    'max_movie_betweenness_node': '作品最大介数中心性节点',
    'max_movie_betweenness_node_names': '作品最大介数中心性节点名称',
    'density': '密度',
    'average_clustering': '平均聚类系数',
    'is_connected': '是否连通',
    'diameter': '直径',
    'num_connected_components': '连通子图数量',
    'largest_cc_num_nodes': '最大连通子图节点数',
    'largest_cc_num_nodes_ratio': '最大连通子图节点比例',
    'largest_cc_num_edges': '最大连通子图边数',
    'largest_cc_num_edges_ratio': '最大连通子图边比例'
}
# 指标说明
metric_description = {
    'works_count': '电影/电视剧数量',
    'director_count': '导演总数',
    'actor_count': '演员总数', 
    'cast_count': '所有影人（包括导演和演员）的总数',
    'rating_max': '所有作品中的最高评分值',
    'rating_min': '所有作品中的最低评分值',
    'rating_mean': '所有作品评分的平均值',
    'rating_people_num': '为保证数据质量，选取评分人数不少于该标准的作品，两者标准可能不同',
    'num_nodes': '网络中节点的总数量',
    'num_edges': '网络中边的总数量',
    'max_degree': '节点中最大的度值（度：节点连接的边数，反映直接影响力）',
    'max_degree_node': '具有最大度值的节点ID',
    'max_degree_node_names': '具有最大度值的节点名称',
    'max_betweenness': '节点中最大的介数中心性值（介数中心性：衡量节点作为桥梁的重要性）',
    'max_betweenness_node': '具有最大介数中心性的节点ID',
    'max_betweenness_node_names': '具有最大介数中心性的节点名称',
    'num_cast_nodes': '影人类型节点的数量',
    'max_cast_degree': '影人节点中最大的度值（度：节点连接的边数，这里是影人参与的作品数量）',
    'max_cast_degree_node': '具有最大度值的影人节点ID',
    'max_cast_degree_node_names': '具有最大度值的影人姓名',
    'max_cast_betweenness': '影人节点中最大的介数中心性值（介数中心性：衡量节点作为桥梁的重要性）',
    'max_cast_betweenness_node': '具有最大介数中心性的影人节点ID',
    'max_cast_betweenness_node_names': '具有最大介数中心性的影人姓名',
    'num_movie_nodes': '作品类型节点的数量',
    'max_movie_degree': '作品节点中最大的度值（度：节点连接的边数，反映合作影人数量）',
    'max_movie_degree_node': '具有最大度值的作品节点ID',
    'max_movie_degree_node_names': '具有最大度值的作品名称',
    'max_movie_betweenness': '作品节点中最大的介数中心性值（介数中心性：衡量节点在网络连通中的关键程度）',
    'max_movie_betweenness_node': '具有最大介数中心性的作品节点ID',
    'max_movie_betweenness_node_names': '具有最大介数中心性的作品名称',
    'density': '网络的密度，表示实际边数与可能最大边数的比值',
    'average_clustering': '网络中所有节点的聚类系数的平均值',
    'is_connected': '判断网络是否连通（即任意两个节点间都存在路径）',
    'diameter': '网络的直径，即所有节点对之间最短路径的最大长度',
    'num_connected_components': '网络中连通子图的数量',
    'largest_cc_num_nodes': '网络中最大连通子图包含的节点数量',
    'largest_cc_num_nodes_ratio': '最大连通子图节点数占网络总节点数的比例',
    'largest_cc_num_edges': '网络中最大连通子图包含的边数量',
    'largest_cc_num_edges_ratio': '最大连通子图边数占网络总边数的比例'
}
stats_parts_dict = {
    'df_stats': '原始数据统计',
    'graph_metrics': '整体网络图指标',
    'largest_connected_component': '最大连通子图指标',
    'movie_bipartite_graph_metrics': '作品二分图指标',
    'movie_bipartite_largest_connected_component': '作品二分图最大连通子图指标',
    'cast_bipartite_graph_metrics': '影人二分图指标',
    'cast_bipartite_largest_connected_component': '影人二分图最大连通子图指标'
}

In [ ]:
# 将两个字典中的指标，按部分合并，之后加入excel的sheet中
# writer = pd.ExcelWriter('movie_network_stats.xlsx', engine='xlsxwriter')
with pd.ExcelWriter('movie_network_stats.xlsx', engine='xlsxwriter') as writer:
    for p in stats_parts_dict.keys():
        rows = []
        for key in top_stats[p].keys():
            row = {}
            row['指标'] = key_to_chinese.get(key, key)
            # row['高分电影网络'] = str(top_stats[p][key])
            # row['低分电影网络'] = str(top_stats_asc[p][key])
            row['2025年影视网络'] = str(year_stats[p][key])
            row['说明'] = metric_description.get(key, '')
            rows.append(row)
        df_part = pd.DataFrame(rows)
        sheet_name = stats_parts_dict[p]
        print(sheet_name)
        df_part.to_excel(writer, sheet_name=sheet_name, index=False)
    # df_part.to_excel(f'movie_network_stats.xlsx', sheet_name=sheet_name, index=False)
    # df_part.to_excel(writer, sheet_name=sheet_name, index=False)


In [ ]:
pd.DataFrame.from_dict(top_stats_asc['graph_metrics'], orient='index', columns=['值']).rename(index=key_to_chinese)

In [ ]:
# 片名转为csv文件
def df_movie_titles_to_csv(df, filename, asc=False):
    movie_list = df.sort_values(by='rating_num', ascending=asc)['k_title'].unique()
    # 转为csv, 10列，每列20个电影
    movie_df = pd.DataFrame()
    num_cols = 5
    num_rows = int(np.ceil(len(movie_list) / num_cols))
    for i in range(num_cols):
        col_data = movie_list[i * num_rows:(i + 1) * num_rows]
        movie_df[f'{i*40}-{(i+1)*40}'] = pd.Series(col_data)
    movie_df.to_csv(filename, index=False)

In [ ]:
df_movie_titles_to_csv(df_movie_top_n, 'movie_top_200.csv')
df_movie_titles_to_csv(df_movie_top_n_asc, 'movie_lowest_200.csv', asc=True)

简单介绍二分图的概念：
二分图是一种将网络节点划分为两个互不相交的集合（如影人集合和电影集合），且所有边只在不同集合节点之间连接的图结构。
具体来说：在影人-电影二分图中，影人节点之间不相连，电影节点之间也不相连，所有的合作关系都通过影人与电影之间的边来表示，这种结构完美地建模了"谁参演了什么电影"的关联关系。
基于二分图投影：我们可以将这种二分网络拆分为两个单模网络——影人合作网络（通过共同参演电影建立连接）和电影关联网络（通过共享影人建立连接），从而分别分析影人间的合作模式和电影间的题材关联。

[向右R][向右R]特别声明：
数据来源于公开的网络数据，旨在探索电影合作网络的结构特征，难免存在遗漏或错误，仅供娱乐，请勿过分解读。
